# 03 Raw parse: the four tables, and mapping by meaning
`SOI` column headers are **XBRL labels**, and the report adds columns dynamically. Map them to canonical fields through the `TAG` table, by the tag they label, and read only the columns you need.

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

release folder: data\raw\2026_08_bdc | fixture mode: False


In [2]:
sub = load.load_sub(RDIR)
print(f"SUB: {len(sub)} current filings (superseded originals with prevrpt=1 are dropped)")
print(sub["form"].value_counts().to_string())

SUB: 174 current filings (superseded originals with prevrpt=1 are dropped)
form
10-Q    173
10-K      1


In [3]:
tag = load.load_tag(RDIR)
std = tag[(tag["custom"] == "0") & tag["tag"].isin(mapping.CANONICAL.values())][["tag", "tlabel"]]
print("Canonical tags and their labels in TAG:")
print(std.to_string(index=False))

Canonical tags and their labels in TAG:
                                  tag                                      tlabel
    InvestmentBasisSpreadVariableRate     Investment, Basis Spread, Variable Rate
    InvestmentBasisSpreadVariableRate     Investment, Basis Spread, Variable Rate
               InvestmentInterestRate                    Investment Interest Rate
               InvestmentInterestRate                    Investment Interest Rate
     InvestmentInterestRatePaidInKind     Investment, Interest Rate, Paid in Kind
     InvestmentInterestRatePaidInKind     Investment, Interest Rate, Paid in Kind
               InvestmentMaturityDate                    Investment Maturity Date
               InvestmentMaturityDate                    Investment Maturity Date
                InvestmentOwnedAtCost                      Investment Owned, Cost
                InvestmentOwnedAtCost                         Adjusted cost basis
           InvestmentOwnedAtFairValue                Inves

In [4]:
header = load.soi_header(RDIR)
meta = [c for c in header if c.islower()]
axes = [c for c in header if c.endswith("Axis")]
vals = [c for c in header if c not in meta + axes]
print("metadata:", meta); print("axes:", axes); print("values:", vals)

metadata: ['adsh', 'cik', 'name', 'ddate', 'qtrs', 'form', 'filed', 'period', 'inlineurl', 'cstm']
axes: ['Industry Sector Axis', 'Investment, Identifier Axis', 'Investment, Issuer Affiliation Axis', 'Investment Type Axis', 'Asset Acquisition Axis', 'Asset Class Axis', 'Award Date Axis', 'Award Type Axis', 'Balance Sheet Location Axis', 'Business Acquisition Axis', 'Cash and Cash Equivalents Axis', 'Class of Financing Receivable Axis', 'Class of Stock Axis', 'Collaborative Arrangement and Arrangement Other than Collaborative Axis', 'Collateral Held Axis', 'Concentration Risk Benchmark Axis', 'Concentration Risk Type Axis', 'Consolidated Entities Axis', 'Consolidation Items Axis', 'Counterparty Name Axis', 'Credit Facility Axis', "Credit Rating, Standard & Poor's Axis", 'Currency Axis', 'Customer Axis', 'Debt Instrument Axis', 'Debt Instrument, Redemption, Period Axis', 'Derivative Instrument Axis', 'Equity Components Axis', 'Equity Interest Type Axis', 'Equity Method Investment, Noncon

In [5]:
colmap = mapping.map_soi_columns(header, tag)
for field, cols in colmap.items():
    print(f"  {field:15s} <- {cols}")
print("Unmapped canonical fields:", set(mapping.CANONICAL) - set(colmap) or "none")
use = mapping.soi_usecols(header, colmap)
print(f"Reading {len(use)} of {len(header)} SOI columns; unmapped value columns to review: {[c for c in vals if c not in use]}")

  fair_value      <- ['Investment Owned, Fair Value', 'Initial fair value of Investment']
  cost            <- ['Investment Owned, Cost', 'Adjusted cost basis']
  principal       <- ['Investment Owned, Balance, Principal Amount']
  interest_rate   <- ['Investment Interest Rate']
  spread          <- ['Investment, Basis Spread, Variable Rate']
  maturity        <- ['Investment Maturity Date']
  pik_rate        <- ['Investment, Interest Rate, Paid in Kind']
  pct_net_assets  <- ['Investment Owned, Net Assets, Percentage']
Unmapped canonical fields: none
Reading 22 of 252 SOI columns; unmapped value columns to review: ['Agreement', 'CreditFacilityUsage', 'ExemptSecurity', 'FinancingReceivableName', 'InterestRatePeriod', 'Investee', 'InvestmentBasisSpreadVariableRate', 'InvestmentHoldings', 'InvestmentInterestRateFloor', 'InvestmentPerformanceStatus', 'InvestmentsClassificationByIndustry', 'InvestmentSecondaryCategorization1', 'InvestmentsIdentifier', 'InvestmentSubType', 'InvestmentVariab

In [6]:
num = load.load_num(RDIR, tags=[reconcile.FV_TAG])        # chunked read, filtered by tag
print(f"NUM fair-value facts: {len(num):,}; dimension counts:")
print(num["dimn"].value_counts().sort_index().to_string())

NUM fair-value facts: 152,244; dimension counts:
dimn
0       324
1    127609
2     13198
3      9476
4      1242
5       395


**Scale.** A full release's `num.tsv` and `soi.tsv` can each be hundreds of megabytes. Read `NUM` in chunks and filter early, and read only the `SOI` columns you use, as these helpers do. DuckDB can also query the TSV files directly; convert to Parquet once if you query repeatedly.

**Your turn.** Complete `docs/schema.md`: the tag and `SOI` column behind every canonical field, and a decision on every unmapped value column (Case Study 2B, Part 1).